# Notebook 3 — Dashboard Power BI & Storytelling RH
**SOLUTION COMPLETE**

**Projet :** TelcomCI — RH & Masse Salariale  
**Prerequis :** Notebook 2 termine, vues et procedure SQL creees  
**Auteur :** DataProjectLab


---
## Contexte

Les analyses SQL des Notebooks 1 et 2 ont produit des reponses chiffrees aux 5 questions de la DRH TelcomCI. Ce Notebook 3 transforme ces analyses en un **dashboard RH 5 pages** accessible a la direction sans connaissance SQL.

**Ce que ce notebook couvre :**
1. Connexion Power BI → SQL Server (Import Mode)
2. Modele de donnees en etoile avec table Calendrier
3. Les 12 mesures DAX avec valeurs de validation
4. Architecture page par page (5 pages)
5. Checklist de validation finale

---
## Etape 1 — Connexion Power BI → SQL Server

### METHODE — Import vs DirectQuery

| | Import | DirectQuery |
|---|---|---|
| Donnees | Copiees dans Power BI | Restent dans SQL Server |
| Vitesse | Rapide (tout en memoire) | Lent (requete SQL a chaque clic) |
| Fraicheur | Refresh manuel ou programme | Toujours a jour |
| DAX Time Intelligence | Toutes mesures disponibles | Limitations |

**Pour TelcomCI : Import.** Les donnees RH 2021-2024 sont historiques. L'Import offre la meilleure experience et toutes les fonctionnalites DAX.

**Procedure de connexion :**
```
Power BI Desktop → Accueil → Obtenir des donnees → SQL Server
Serveur : localhost | Base : TelcomCI_RH | Mode : Import

Charger les VUES (pas les tables brutes) :
  vw_employes_propres
  vw_salaires_valides
  vw_absences_valides

Charger aussi les tables de reference :
  departements
  recrutements
  evaluations
```

> **Pourquoi les vues et non les tables brutes ?**
> Les vues excluent deja les anomalies (salaires negatifs, doublons, nb_jours negatifs). Charger les tables brutes ferait apparaitre ces anomalies dans le dashboard.

**Conversions obligatoires dans Power Query :**
```
vw_employes_propres :
  date_embauche, date_depart → Date
  salaire_base, salaire_net → Nombre decimal

vw_absences_valides :
  date_debut, date_fin → Date
  justifiee, impact_salaire → Nombre entier (BIT devient 0/1)

evaluations :
  note_performance, note_competences, note_attitude, note_globale → Nombre decimal
```

In [ ]:
-- Verifier les vues avant connexion Power BI
SELECT TABLE_NAME, TABLE_TYPE
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'dbo'
ORDER BY TABLE_TYPE DESC, TABLE_NAME;

-- Valeurs de reference a verifier dans Power BI apres connexion
SELECT
    COUNT(DISTINCT employe_id)                          AS total_employes_propres,
    COUNT(CASE WHEN statut = 'Actif' THEN 1 END)        AS employes_actifs,
    COUNT(CASE WHEN statut = 'Inactif' THEN 1 END)      AS total_departs,
    ROUND(AVG(CAST(salaire_base AS FLOAT)), 0)          AS sal_base_moyen
FROM vw_employes_propres;

-- Resultats attendus :
-- total_employes_propres : 506
-- employes_actifs        : 446 (apres correction doublons sur vue)
-- total_departs          : 60
-- sal_base_moyen         : ~1 028 000 FCFA


---
## Etape 2 — Modele de donnees en etoile

### METHODE — Pourquoi le schema en etoile ?

Power BI est optimise pour le **schema en etoile** : une table de faits centrale entouree de tables de dimension. Ce schema permet au moteur VertiPaq de compresser et interroger les donnees 10 a 100x plus vite qu'un schema normalise.

```
       departements (dim)           Calendrier (dim temps)
             |                              |
             | dept_id                      | date
             |                              |
vw_employes_propres (dim employes)           |
    employe_id  dept_id                      |
         |          |                        |
         |     recrutements (fait)           |
         |                                   |
  vw_salaires_valides (FAIT principal) ──────+
  vw_absences_valides (fait)
  evaluations (fait)
```

### Relations a creer dans Power BI (vue Modele)

| De | Vers | Cle | Cardinalite | Direction |
|---|---|---|---|---|
| departements[departement_id] | vw_employes_propres[departement_id] | dept_id | 1:N | Single |
| vw_employes_propres[employe_id] | vw_salaires_valides[employe_id] | emp_id | 1:N | Single |
| vw_employes_propres[employe_id] | vw_absences_valides[employe_id] | emp_id | 1:N | Single |
| vw_employes_propres[employe_id] | evaluations[employe_id] | emp_id | 1:N | Single |
| departements[departement_id] | recrutements[departement_id] | dept_id | 1:N | Single |
| Calendrier[Date] | vw_salaires_valides[date_cal] | date | 1:N | Single |

> **Regle d'or :** toujours direction Single (→). La direction Both (↔) peut creer des ambiguites de calcul et ralentir les requetes DAX. Si un visuel ne filtre pas comme attendu, verifier d'abord la direction des relations.

In [ ]:
-- Formule DAX : table Calendrier (Accueil → Nouvelle table)
Calendrier =
ADDCOLUMNS(
    CALENDAR(DATE(2021,1,1), DATE(2024,12,31)),
    "Annee",        YEAR([Date]),
    "Mois_Num",     MONTH([Date]),
    "Mois_Nom",     FORMAT([Date], "MMM"),
    "Mois_Court",   FORMAT([Date], "MMM YYYY"),
    "Trimestre",    "T" & QUARTER([Date]),
    "Semestre",     IF(MONTH([Date]) <= 6, "S1", "S2"),
    "Jour_Semaine", FORMAT([Date], "ddd"),
    "Num_Jour",     WEEKDAY([Date], 2),
    "Est_Weekend",  IF(WEEKDAY([Date], 2) >= 6, 1, 0)
)

-- Apres creation :
-- Clic droit sur la table Calendrier → Marquer comme table de dates → colonne Date
-- OBLIGATOIRE pour que PREVIOUSMONTH() et DATESYTD() fonctionnent


### METHODE — Colonne de date calculee dans vw_salaires_valides

La table `vw_salaires_valides` contient `annee` et `mois` separement, pas de colonne DATE. Il faut creer une colonne calculee dans Power BI pour etablir la relation avec le Calendrier :

```
Dans vw_salaires_valides → Nouvelle colonne :
date_cal = DATE(vw_salaires_valides[annee], vw_salaires_valides[mois], 1)
```

Cette colonne calcule le 1er jour de chaque mois (01/2021, 02/2021, etc.) et permet la jointure avec `Calendrier[Date]`.

---
## Etape 3 — Les 12 mesures DAX

### METHODE — La table _Mesures

Creer une table vide nommee `_Mesures` qui contiendra toutes les mesures :
```
Accueil → Entrer des donnees → Nommer '_Mesures' → Charger
Puis supprimer la colonne automatique 'Colonne1'
```
Le `_` place la table en tete de la liste alphabetique dans le panneau Champs.

In [ ]:
-- ════════════════════════════════════════
-- MESURES DE BASE
-- ════════════════════════════════════════

-- Mesure 1 : Effectif actif
Effectif Actif =
CALCULATE(
    COUNTROWS(vw_employes_propres),
    vw_employes_propres[statut] = "Actif"
)
-- Valeur attendue sans filtre : 446

-- Mesure 2 : Masse Salariale Brute (brut + primes)
Masse Salariale Brute =
SUM(vw_salaires_valides[salaire_brut])
+ SUM(vw_salaires_valides[prime])
-- Valeur attendue sans filtre toute periode : 17 918 364 926 FCFA
-- Valeur attendue filtre juin 2024 : 568 593 915 FCFA

-- Mesure 3 : Salaire Net Moyen
Salaire Net Moyen =
AVERAGE(vw_salaires_valides[salaire_net])
-- Valeur attendue sans filtre : 869 250 FCFA

-- Mesure 4 : Taux de Turnover
Taux Turnover =
DIVIDE(
    CALCULATE(
        COUNTROWS(vw_employes_propres),
        NOT ISBLANK(vw_employes_propres[date_depart])
    ),
    COUNTROWS(vw_employes_propres)
)
-- Valeur attendue sans filtre : 11.9%

-- Mesure 5 : Note Performance Moyenne
Note Performance Moyenne =
AVERAGE(evaluations[note_globale])
-- Valeur attendue sans filtre : 3.83


In [ ]:
-- ════════════════════════════════════════
-- MESURES AVANCEES
-- ════════════════════════════════════════

-- Mesure 6 : Taux Absenteisme
Taux Absenteisme =
VAR _jours_abs   = SUM(vw_absences_valides[nb_jours])
VAR _nb_employes = [Effectif Actif]
VAR _nb_jours_cal = COUNTROWS(
    FILTER(
        DISTINCT(Calendrier[Date]),
        WEEKDAY(Calendrier[Date], 2) < 6  -- jours ouvrables uniquement
    )
)
RETURN
DIVIDE(_jours_abs, _nb_employes * _nb_jours_cal)
-- Valeur attendue ~0.60% sur toute la periode

-- Mesure 7 : Taux Absences Injustifiees
Taux Absences Injustifiees =
DIVIDE(
    CALCULATE(
        SUM(vw_absences_valides[nb_jours]),
        vw_absences_valides[justifiee] = 0
    ),
    SUM(vw_absences_valides[nb_jours])
)
-- Valeur attendue sans filtre : 17.5%

-- Mesure 8 : Cout Recrutement Moyen
Cout Recrutement Moyen =
AVERAGE(recrutements[cout_recrutement])
-- Valeur attendue sans filtre : 1 365 663 FCFA

-- Mesure 9 : Cout Turnover Estime (6 mois de salaire par depart)
Cout Turnover Estime =
VAR _departs =
    CALCULATE(
        COUNTROWS(vw_employes_propres),
        NOT ISBLANK(vw_employes_propres[date_depart])
    )
VAR _sal_moy = [Salaire Net Moyen]
RETURN _departs * _sal_moy * 6
-- Valeur attendue sans filtre : ~312 930 000 FCFA (~313 M FCFA)

-- Mesure 10 : Masse Salariale Mois Precedent
Masse Sal Mois Prec =
CALCULATE(
    [Masse Salariale Brute],
    PREVIOUSMONTH(Calendrier[Date])
)
-- Necessite que Calendrier soit marque comme table de dates

-- Mesure 11 : Variation Masse Salariale
Variation Masse Salariale =
VAR _actuel = [Masse Salariale Brute]
VAR _prec   = [Masse Sal Mois Prec]
RETURN
DIVIDE(_actuel - _prec, _prec)
-- Exemple juin 2024 vs mai 2024 : +22.8% (mois de primes)

-- Mesure 12 : Quadrant Performance
Quadrant Performance =
VAR _note = [Note Performance Moyenne]
VAR _sal  = [Salaire Net Moyen]
VAR _med_note = 3.83
VAR _med_sal  = 914943
RETURN
SWITCH(TRUE(),
    _note >= _med_note && _sal >= _med_sal, "Top Performer Bien Paye",
    _note >= _med_note && _sal <  _med_sal, "Top Performer Sous-Paye",
    _note <  _med_note && _sal >= _med_sal, "Faible Perf Bien Paye",
    "A Accompagner"
)


### INTERPRETATION — Mesures DAX

**Tableau de validation complet (sans filtre) :**

| Mesure | Valeur attendue |
|---|---|
| [Effectif Actif] | **446** |
| [Masse Salariale Brute] totale | **17 918 364 926 FCFA** |
| [Masse Salariale Brute] juin 2024 | **568 593 915 FCFA** |
| [Salaire Net Moyen] | **869 250 FCFA** |
| [Taux Turnover] | **11.9%** |
| [Note Performance Moyenne] | **3.83** |
| [Taux Absences Injustifiees] | **17.5%** |
| [Cout Recrutement Moyen] | **1 365 663 FCFA** |
| [Cout Turnover Estime] | **~313 M FCFA** |
| [Variation Masse Sal.] juin/mai 2024 | **+22.8%** |

**Sur le Quadrant Performance :**
Les seuils `_med_note = 3.83` et `_med_sal = 914 943` sont les medianes calculees en SQL (NB2 Etape 4). Pour les rendre dynamiques selon la periode filtree, on peut remplacer les constantes par `MEDIANX(...)` mais cela ralentit la mesure — les constantes sont suffisantes pour ce dashboard.

> **Si une mesure retourne BLANK ou une valeur inattendue :**
> 1. Verifier que la table Calendrier est marquee comme table de dates
> 2. Verifier la colonne `date_cal` dans vw_salaires_valides
> 3. Verifier que la relation Calendrier → vw_salaires_valides est sur `date_cal`
> 4. Verifier que `justifiee` a ete converti en Nombre entier (0/1) dans Power Query

---
## Etape 4 — Architecture des 5 pages

### Page 1 — Vue DRH

**Objectif :** donner a la DRH une vue d'ensemble en 30 secondes.

**Visuels :**

**1. Bandeau 4 cartes KPI** (haut de page)
```
[Effectif Actif]   [Taux Turnover]   [Note Perf. Moy.]   [Cout Turnover Estime]
446                 11.9%              3.83 / 5             ~313 M FCFA
```

**2. Barres horizontales : effectif actif vs cible par departement**
- Y : departements[nom]
- X1 : [Effectif Actif] (barres bleues)
- X2 : departements[effectif_cible] (ligne de reference orange pointillee)
- Tous les departements sont sous la cible — visualisation du sous-effectif global

**3. Courbe : evolution masse salariale mensuelle 2022-2024**
- X : Calendrier[Mois_Court]
- Y : [Masse Salariale Brute]
- Legende : departements[nom]
- Valeurs pics decembre : entre +100% et +110% (primes annuelles)

**4. Anneau : repartition par type de contrat**
- CDI : **415 (81.4%)**, CDD : 71 (13.9%), Interim : 24 (4.7%)

**5. Slicers :** Calendrier[Annee], departements[nom]

In [ ]:
-- Requete de validation Page 1
SELECT
    COUNT(CASE WHEN statut = 'Actif' THEN 1 END)          AS effectif_actif,
    COUNT(CASE WHEN statut = 'Inactif' THEN 1 END)        AS total_departs,
    ROUND(COUNT(CASE WHEN statut = 'Inactif' THEN 1 END)
          * 100.0 / COUNT(*), 1)                          AS taux_turnover_pct,
    type_contrat,
    COUNT(*)                                               AS nb_par_contrat
FROM vw_employes_propres
GROUP BY type_contrat WITH ROLLUP
ORDER BY nb_par_contrat DESC;

-- Valeurs attendues :
-- CDI     : 415 (81.4%)
-- CDD     : 71  (13.9%)
-- Interim : 24  (4.7%)
-- TOTAL   : 510 brut → 506 apres nettoyage


### Page 2 — Masse Salariale

**Objectif :** analyser la structure et l'evolution du cout salarial.

**1. Aires empilees : brut + primes par departement et par mois**
- X : Calendrier[Mois_Court]
- Y : [Masse Salariale Brute]
- Legende : departements[nom]
- Les pics de decembre (primes) sont clairement visibles

**2. Barres groupees : masse brute vs masse nette par departement**
- Valeurs juin 2024 (top 3) :
  - Technique & Reseau : **127.6 M FCFA** brut
  - Service Client : **108.4 M FCFA** brut
  - Commercial & Ventes : **101.0 M FCFA** brut

**3. Carte KPI avec fleche : [Variation Masse Salariale]**
- Juin 2024 vs mai 2024 : **+22.8%**
- Afficher en vert si > 0, rouge si < 0

**4. Tableau : top 10 employes par salaire brut actuel**
- Colonnes : Employe, Departement, Poste, Salaire brut, Salaire net
- Trier par salaire brut decroissant

In [ ]:
-- Requete de validation Page 2
SELECT
    d.nom                                               AS departement,
    ROUND(SUM(s.salaire_brut + s.prime), 0)             AS masse_brute_juin_2024,
    ROUND(SUM(s.salaire_net), 0)                        AS masse_nette_juin_2024,
    ROUND(SUM(s.prime), 0)                              AS primes_juin_2024
FROM vw_salaires_valides s
JOIN departements d ON s.departement_id = d.departement_id
WHERE s.annee = 2024 AND s.mois = 6
GROUP BY d.departement_id, d.nom
ORDER BY masse_brute_juin_2024 DESC;

-- Valeurs attendues (top 5) :
-- Technique & Reseau    : 127 644 756 FCFA
-- Service Client        : 108 365 936 FCFA
-- Commercial & Ventes   : 101 006 124 FCFA
-- Informatique          :  72 172 263 FCFA
-- Finance               :  36 273 550 FCFA
-- TOTAL juin 2024       : 568 593 915 FCFA


### Page 3 — Turnover & Departs

**Objectif :** identifier les departements et motifs a risque.

**1. Barres horizontales : taux turnover par departement**
- Ligne de reference a 15% (seuil d'alerte standard)
- Couleur conditionnelle : rouge si > 15%, orange si 10-15%, vert si < 10%
- Marketing 23.3% et Logistique 16.0% sont au-dessus du seuil

**2. Anneau : repartition des motifs de depart**
- Demission, Fin CDD, Retraite, Licenciement, Mutation
- Tous les motifs sont relativement equilibres (~20% chacun)

**3. Grande carte KPI rouge : [Cout Turnover Estime]**
- **~313 M FCFA** sur toute la periode
- Sous-titre : '6 mois de salaire moyen x 60 departs'

**4. Courbe : evolution du turnover annuel par departement**
- X : vw_employes_propres[annee_depart] (colonne calculee)
- Y : [Taux Turnover]
- Mise en evidence Marketing et Finance — les plus en hausse en 2023

In [ ]:
-- Requete de validation Page 3
SELECT
    d.nom                                               AS departement,
    COUNT(*)                                            AS nb_departs,
    e.motif_depart,
    ROUND(COUNT(*) * 100.0
          / SUM(COUNT(*)) OVER (PARTITION BY d.nom), 1) AS pct_motif_dept
FROM employes e
JOIN departements d ON e.departement_id = d.departement_id
WHERE e.date_depart IS NOT NULL
GROUP BY d.nom, e.motif_depart
ORDER BY d.nom, nb_departs DESC;

-- Verifier que le cout turnover Power BI correspond
SELECT
    COUNT(*) AS nb_departs_total,
    ROUND(AVG(s.salaire_net), 0) AS sal_net_moyen,
    ROUND(COUNT(*) * AVG(s.salaire_net) * 6, 0) AS cout_turnover_estime
FROM employes e
JOIN vw_salaires_valides s ON e.employe_id = s.employe_id
WHERE e.date_depart IS NOT NULL;

-- Resultat attendu :
-- nb_departs_total     : 60
-- sal_net_moyen        : ~869 250 FCFA
-- cout_turnover_estime : ~312 930 000 FCFA


### Page 4 — Absenteisme

**Objectif :** detecter les departements et periodes problematiques.

**1. Matrice (heatmap) : jours absence par departement x mois 2023**
- Lignes : departements[nom]
- Colonnes : Calendrier[Mois_Num]
- Valeurs : SUM(vw_absences_valides[nb_jours])
- Mise en forme conditionnelle par regles :
  - 0 : blanc
  - 1-10 : bleu pale
  - 11-20 : orange
  - > 20 : rouge

**Valeurs 2023 attendues dans la heatmap :**

| Departement | Mois le plus charge | Nb jours |
|---|---|---|
| Service Client | Oct (29j) et Sep (27j) | **Alerte rouge** |
| Technique & Reseau | Aout (31j) | **Alerte rouge** |
| Informatique | Janv (17j), Fev (21j), Dec (21j) | Orange |
| Commercial | Fev (23j), Aout (23j) | Orange |

**2. Barres : taux absences injustifiees par departement**
- Valeur globale : **17.5%** des absences sont injustifiees
- Ligne de reference a 15%

**3. Carte KPI : [Taux Absences Injustifiees]**
- **17.5%** → au-dessus de la norme acceptable (< 15%)

**4. Tableau : top 10 employes avec le plus de jours d'absence injustifiee**
- Filtrer sur justifiee = 0 dans le tableau

In [ ]:
-- Requete de validation Page 4 — heatmap absenteisme 2023
SELECT *
FROM (
    SELECT
        d.nom                      AS departement,
        MONTH(a.date_debut)        AS mois,
        a.nb_jours
    FROM vw_absences_valides a
    JOIN departements d ON a.departement_id = d.departement_id
    WHERE YEAR(a.date_debut) = 2023
) src
PIVOT (
    SUM(nb_jours)
    FOR mois IN ([1],[2],[3],[4],[5],[6],[7],[8],[9],[10],[11],[12])
) pvt
ORDER BY [9] + [10] DESC;  -- trier par mois les plus charges

-- Valeurs les plus elevees en 2023 :
-- Service Client      Oct=29, Sep=27, Jun=23 → Alerte critique
-- Technique & Reseau  Aou=31                 → Alerte critique
-- Informatique        Fev=21, Dec=21         → Surveillance

-- Taux absenteisme injustifiee global
SELECT
    ROUND(
        SUM(CASE WHEN justifiee = 0 THEN nb_jours ELSE 0 END) * 100.0
        / NULLIF(SUM(nb_jours), 0)
    , 1) AS pct_injustifiees_global
FROM vw_absences_valides;
-- Resultat attendu : 17.5%


### Page 5 — Performance & Recrutement

**Objectif :** identifier les profils a risque de depart et optimiser le recrutement.

**1. Scatter : Note performance x Salaire net par employe**
- X : [Salaire Net Moyen]
- Y : [Note Performance Moyenne]
- Couleur : [Quadrant Performance] (4 couleurs pour les 4 quadrants)
- Lignes de reference : x = 914 943 FCFA (mediane sal.) | y = 3.83 (mediane note)
- Taille bulle : constant (1 point = 1 employe)

**4 quadrants et leurs effectifs :**
- Top Performer Bien Paye : **112 employes** — conserver, fideliser
- **Top Performer Sous-Paye : 114 employes** — ALERTE, risque depart eleve
- Faible Perf Bien Paye : **108 employes** — revoir les missions
- A Accompagner : **106 employes** — plan de developpement

**2. Barres horizontales : cout par embauche par canal**
- Trier par cout_par_embauche croissant (meilleur en haut)
- Cabinet : **1 713 517 FCFA** (meilleur)
- Jobboard : **2 420 353 FCFA** (moins efficace)

**3. Tableau : alerte RH — Top Performers Sous-Payes (note >= 4.0 + Q1 sal.)**
- **35 employes** a risque eleve de depart
- Colonnes : Employe, Departement, Poste, Salaire net, Note globale, Recommandation

**4. Anneau : repartition des recommandations 2023**
- Statu quo : **460 (37.7%)**, Formation : 318 (26.0%),
  Augmentation : 225 (18.4%), Attention : 111 (9.1%), Promotion : 107 (8.8%)

In [ ]:
-- Requete de validation Page 5
-- 1. Synthese quadrants (doit correspondre exactement au scatter)
EXEC sp_rapport_rh_departement @dept_id = 'DEP02', @annee = 2023;

-- 2. Top performers sous-payes (alerte RH — 35 employes)
WITH eval_2023 AS (
    SELECT employe_id, note_globale, recommandation
    FROM evaluations WHERE annee = 2023
),
sal_juin24 AS (
    SELECT employe_id, ROUND(AVG(salaire_net), 0) AS sal_net
    FROM vw_salaires_valides WHERE periode = '2024-06'
    GROUP BY employe_id
),
quartiles AS (
    SELECT
        e.employe_id,
        e.prenom + ' ' + e.nom  AS employe,
        d.nom                   AS departement,
        e.poste, sa.sal_net,
        ev.note_globale,
        ev.recommandation,
        NTILE(4) OVER (ORDER BY sa.sal_net ASC) AS q_sal
    FROM vw_employes_propres e
    JOIN departements d    ON e.departement_id = d.departement_id
    LEFT JOIN eval_2023 ev ON e.employe_id     = ev.employe_id
    LEFT JOIN sal_juin24 sa ON e.employe_id    = sa.employe_id
    WHERE e.statut = 'Actif'
      AND ev.note_globale IS NOT NULL
      AND sa.sal_net IS NOT NULL
)
SELECT employe, departement, poste, sal_net, note_globale, recommandation
FROM quartiles
WHERE q_sal = 1 AND note_globale >= 4.0
ORDER BY note_globale DESC;
-- Resultat attendu : 35 lignes
-- Note moy : 4.21 | Salaire moy net : 622 403 FCFA

-- 3. Recommandations 2023
SELECT
    recommandation,
    COUNT(*)                                           AS nb,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM evaluations
WHERE annee = 2023
GROUP BY recommandation
ORDER BY nb DESC;
-- Statu quo : 460 (37.7%) | Formation : 318 (26.0%) | Augmentation : 225 (18.4%)


---
## Etape 5 — Sous-titres dynamiques

### METHODE — SELECTEDVALUE()

`SELECTEDVALUE(table[colonne], 'valeur_par_defaut')` retourne la valeur selectionnee dans un slicer. Si aucun filtre ou plusieurs valeurs : retourne la valeur par defaut. C'est le pattern pour les titres adaptatifs.

In [ ]:
-- Sous-titres dynamiques pour chaque page

Sous Titre Page 1 =
VAR _dept  = SELECTEDVALUE(departements[nom],    "Tous les departements")
VAR _annee = SELECTEDVALUE(Calendrier[Annee],    "2021 - 2024")
RETURN
"TelcomCI RH  ·  " & _dept & "  ·  " & _annee

Sous Titre Page 2 =
VAR _ms    = FORMAT([Masse Salariale Brute] / 1000000, "#,##0.0") & " M FCFA"
VAR _var   = FORMAT([Variation Masse Salariale], "+0.0%;-0.0%;0.0%")
RETURN
"Masse salariale : " & _ms & "  ·  Variation : " & _var & " vs mois prec."

Sous Titre Page 3 =
VAR _turn  = FORMAT([Taux Turnover], "0.0%")
VAR _cout  = FORMAT([Cout Turnover Estime] / 1000000, "#,##0") & " M FCFA"
RETURN
"Taux turnover : " & _turn & "  ·  Cout estime : " & _cout

Sous Titre Page 4 =
VAR _inj   = FORMAT([Taux Absences Injustifiees], "0.0%")
RETURN
"Absences injustifiees : " & _inj & " du total  ·  Cible < 15%"

Sous Titre Page 5 =
VAR _quad  = SELECTEDVALUE(evaluations[recommandation], "Toutes recommandations")
RETURN
"Recommandation : " & _quad & "  ·  Mediane note 3.83  ·  Mediane sal. 914 943 FCFA"


---
## Etape 6 — Checklist de validation finale

**Procedure :** ouvrir SSMS et Power BI cote a cote, enlever tous les filtres Power BI, et comparer chaque valeur.

| Mesure DAX | Valeur attendue (sans filtre) | Statut |
|---|---|---|
| [Effectif Actif] | **446** | [ ] |
| [Masse Salariale Brute] juin 2024 | **568 593 915 FCFA** | [ ] |
| [Salaire Net Moyen] | **869 250 FCFA** | [ ] |
| [Taux Turnover] | **11.9%** | [ ] |
| [Note Performance Moyenne] | **3.83** | [ ] |
| [Taux Absences Injustifiees] | **17.5%** | [ ] |
| [Cout Recrutement Moyen] | **1 365 663 FCFA** | [ ] |
| [Cout Turnover Estime] | **~313 M FCFA** | [ ] |
| [Effectif Actif] filtre DEP04 Service Client | **115** | [ ] |
| [Note Perf. Moy.] filtre DEP08 Informatique | **3.91** | [ ] |

### Erreurs les plus frequentes

| Erreur | Cause probable | Solution |
|---|---|---|
| PREVIOUSMONTH retourne BLANK | Calendrier non marque comme table de dates | Clic droit Calendrier → Marquer |
| Effectif Actif incorrect | Filtre statut non applique | Verifier la condition `statut = 'Actif'` |
| Scatter vide page 5 | Relation evaluations → employes manquante | Ajouter relation employe_id |
| Heatmap ne filtre pas par dept | Direction Both sur relation dept | Passer en Single direction |
| Taux Absenteisme trop eleve | nb_jours non filtre (negatifs inclus) | Charger vw_absences_valides |


In [ ]:
-- Requete SQL finale de validation globale
-- Ces chiffres doivent correspondre exactement aux cartes KPI de la Page 1

SELECT
    COUNT(CASE WHEN statut = 'Actif' THEN 1 END)            AS effectif_actif,
    COUNT(CASE WHEN statut = 'Inactif' THEN 1 END)          AS nb_departs,
    ROUND(COUNT(CASE WHEN statut='Inactif' THEN 1 END)
          * 100.0 / COUNT(*), 1)                            AS taux_turnover_pct
FROM vw_employes_propres;

-- Masse salariale juin 2024
SELECT
    ROUND(SUM(salaire_brut + prime), 0)                     AS masse_brute_juin_2024,
    ROUND(AVG(salaire_net), 0)                              AS sal_net_moyen,
    COUNT(DISTINCT employe_id)                              AS employes_payes
FROM vw_salaires_valides
WHERE annee = 2024 AND mois = 6;

-- Note performance et taux injustifiees
SELECT
    ROUND(AVG(note_globale), 2)                             AS note_perf_moy
FROM evaluations WHERE annee = 2023;

SELECT
    ROUND(SUM(CASE WHEN justifiee=0 THEN nb_jours ELSE 0 END) * 100.0
          / NULLIF(SUM(nb_jours),0), 1)                     AS taux_injustifiees_pct
FROM vw_absences_valides;

-- Resultats attendus :
-- effectif_actif       : 446
-- taux_turnover_pct    : 11.9%
-- masse_brute_juin2024 : 568 593 915 FCFA
-- sal_net_moyen        : 869 250 FCFA
-- note_perf_moy        : 3.83
-- taux_injustifiees    : 17.5%


---
## Bilan du Notebook 3

### Ce que le dashboard TelcomCI RH permet de piloter

| Page | Question DRH repondue |
|---|---|
| 1 — Vue DRH | Quel est l'effectif actif par dept ? Quelle masse salariale ce mois ? |
| 2 — Masse Salariale | Comment evolue le cout salarial ? Quels departements pesent le plus ? |
| 3 — Turnover | Ou sont les risques de depart ? Combien ca coute ? |
| 4 — Absenteisme | Quels departements sont en tension operationnelle ? |
| 5 — Performance | Qui sont les employes a risque ? Quel canal de recrutement privilegier ? |

### Les 4 alertes prioritaires issues du dashboard

**1. 114 Top Performers Sous-Payes** — dont 35 en Q1 salaire avec note >= 4.0. Ces employes sont les cibles privilegiees des recruteurs externes. Budget d'augmentation cible recommande.

**2. Service Client en tension critique** — sous-effectif -14 vs cible, absenteisme en hausse +65j en 2023 (oct : 29j), turnover 10.8%. Recrutement urgent de 10-15 agents et enquete de satisfaction.

**3. Marketing 23.3% de turnover** — 1 employe sur 4 parti sur la periode. Motif principal : retraites non remplaces. Plan de succession a mettre en place.

**4. Jobboard a abandonner** — cout par embauche 2 420 353 FCFA soit +41% plus couteux que le Cabinet. Transferer le budget Jobboard vers Cabinet et Referencement.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structures et orientes metier.